# Intensity calibration (#33)

Log energy distributions of each source in `silver.flashes`, and the mapping of the current
calibration ([methodology](../methodology.md#flash-intensity)).

**Re-run.** From `data/`, with `SCINTILLE_LAKE_*` pointing at the lake, set `START` and `END` below
(the calibration window, `window` in the JSON file), then execute this notebook in an environment
with Matplotlib (`uv run --with matplotlib --with jupyter jupyter nbconvert --execute --to notebook
--inplace ../docs/analysis/intensity-calibration.ipynb`). Fitting itself is
`uv run scintille-ingest calibrate --start ... --end ...`.

In [ ]:
from datetime import date

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scintille_ingest.intensity import CURRENT, EXCLUDED_FLAGS, intensity
from scintille_lake.connect import duckdb_attach

START, END = CURRENT.window or (date(2026, 10, 5), date(2026, 10, 12))
SAMPLE = 200_000
COLORS = {"glm-east": "#2a78d6", "glm-west": "#eb6834", "mtg-li": "#1baf7a"}

In [ ]:
connection = duckdb.connect()
duckdb_attach(connection)
flags = " AND ".join(f"NOT list_contains(quality, '{f}')" for f in EXCLUDED_FLAGS)
flashes = pl.from_arrow(
    connection.execute(
        f"""
        SELECT source, energy_raw, energy_unit FROM lake.silver.flashes
        WHERE event_time >= ? AND event_time < ? AND energy_raw > 0 AND {flags}
        USING SAMPLE reservoir({SAMPLE} ROWS) REPEATABLE (33)
        """,
        [START, END],
    ).to_arrow_table()
)
assert isinstance(flashes, pl.DataFrame)
flashes = flashes.with_columns(log_energy=pl.col("energy_raw").log10())
flashes.group_by("source").agg(
    pl.len().alias("sampled"),
    pl.col("energy_unit").first(),
    *[pl.col("log_energy").quantile(q).alias(f"p{q * 100:g}") for q in (0.01, 0.5, 0.99, 0.999)],
).sort("source")

## Distributions

Log energy per source. The shapes, more than the positions, decide how well percentile matching
holds: a similar shape means a near-linear map between sources.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for source, group in flashes.group_by("source"):
    values = group["log_energy"].to_numpy()
    standard = (values - np.median(values)) / np.subtract(*np.percentile(values, [75, 25]))
    axes[0].hist(values, bins=120, histtype="step", color=COLORS[source[0]], label=source[0])
    axes[1].hist(standard, bins=120, histtype="step", density=True, color=COLORS[source[0]])
axes[0].set(xlabel="log10 energy_raw (unit of each source)", ylabel="flashes")
axes[1].set(xlabel="log energy, median 0, interquartile range 1", ylabel="density")
axes[0].legend()
fig.tight_layout()

## Mapping

Intensity as a function of percentile within each source: the curves should overlap (same
percentile, same intensity), and the reference stays linear in log energy.

In [ ]:
flashes = flashes.with_columns(intensity(flashes, CURRENT))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
grid = np.linspace(0.5, 99.9, 200)
for source, group in flashes.group_by("source"):
    values = group["log_energy"].to_numpy()
    q = np.percentile(values, grid)
    probe = pl.DataFrame(
        {"source": source[0], "energy_raw": 10**q, "energy_unit": group["energy_unit"][0]}
    )
    axes[0].plot(grid, intensity(probe, CURRENT), color=COLORS[source[0]], label=source[0])
    color = COLORS[source[0]]
    axes[1].hist(group["intensity"], bins=50, histtype="step", density=True, color=color)
axes[0].set(xlabel="percentile within source", ylabel="intensity", title=CURRENT.version)
axes[1].set(xlabel="intensity", ylabel="density")
axes[0].legend()
fig.tight_layout()